# 🫀 PULSE ECG — TCN-BiLSTM-Attention Training
**Runtime → Change runtime type → GPU (T4)** before running!

In [ ]:
# ── STEP 1: Upload zip & setup ─────────────────────────────────
from google.colab import files
import zipfile, os, shutil

print("📤 Upload pulse_ecg_upload.zip ...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')

os.chdir('/content/project')

# Fix folder structure: move 'processed' → 'data/processed'
os.makedirs('data/processed', exist_ok=True)
if os.path.exists('processed'):
    for f in os.listdir('processed'):
        src = os.path.join('processed', f)
        dst = os.path.join('data/processed', f)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    print("  Moved processed/ → data/processed/")

for d in ['models/tfjs_model', 'logs', 'evaluation', 'checkpoints']:
    os.makedirs(d, exist_ok=True)

# Verify
print("\n📁 Project files:", [f for f in os.listdir('.') if not f.startswith('.')])
print("📁 data/processed:", os.listdir('data/processed'))
for f in os.listdir('data/processed'):
    size = os.path.getsize(f'data/processed/{f}') / (1024*1024)
    print(f"   {f}: {size:.1f} MB")
print("\n✅ Setup complete!")

In [ ]:
# ── STEP 2: Install dependencies ───────────────────────────────
!pip install -q wfdb scipy scikit-learn matplotlib seaborn tqdm PyYAML h5py pandas
print("✅ Dependencies installed")

In [ ]:
# ── STEP 3: Verify GPU ─────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {gpus}")
if gpus:
    print("✅ GPU ready!")
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → GPU")

In [ ]:
# ── STEP 4: TRAIN 🚀 ──────────────────────────────────────────
!python train.py --skip-download

In [ ]:
# ── STEP 5: View results ──────────────────────────────────────
import json, glob
from IPython.display import Image, display

for ef in sorted(glob.glob('evaluation/*.json')):
    print(f"\n{'='*50}")
    print(f"📊 {os.path.basename(ef)}")
    with open(ef) as f:
        data = json.load(f)
        for k, v in data.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")

for pf in sorted(glob.glob('evaluation/*.png')):
    display(Image(filename=pf, width=500))

In [ ]:
# ── STEP 6: Export to TF.js ────────────────────────────────────
!pip install -q tensorflowjs
!python convert_to_tfjs.py --metadata

In [ ]:
# ── STEP 7: Download trained model ─────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('/content/pulse_trained_model', 'zip', '.', 'models')
size = os.path.getsize('/content/pulse_trained_model.zip') / (1024*1024)
print(f"📦 Model zip: {size:.1f} MB")
files.download('/content/pulse_trained_model.zip')